In [ ]:
!pip install --upgrade gspread gspread_dataframe google-auth --quiet

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import gspread
import pandas as pd
import numpy as np
import itertools
import math

from google.auth import default
from gspread_dataframe import get_as_dataframe, set_with_dataframe

creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# Use your Google Sheet URL
sheet_url = "https://docs.google.com/spreadsheets/d/1nYWjEL8nil-iwyIgmbaWzioOtrGvuzEN5rGYUL7hWIE/edit?gid=318189471#gid=318189471"
spreadsheet = gc.open_by_url(sheet_url)

In [ ]:
location_df = get_as_dataframe(spreadsheet.worksheet("Location_file")).dropna(how='all')
distance_df = get_as_dataframe(spreadsheet.worksheet("Distance_Matrix")).dropna(how='all')
time_df = get_as_dataframe(spreadsheet.worksheet("Time_Matrix")).dropna(how='all')

# Save locally for your script to read
location_df.to_csv("Location_file.csv", index=False)
distance_df.to_csv("Distance_Matrix.csv", index=False)
time_df.to_csv("Time_Matrix.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import itertools
import math
import sys

# Increase recursion depth for global combinatorial search
sys.setrecursionlimit(5000)

# =============================================================================
# I. Configuration & Cost Parameters
# =============================================================================
max_hops = 4
travel_time_threshold = 100000
max_allowed_combinations = 20000000

VEHICLE_COST_PER_KM = {
    40: 70, 32: 50, 24: 35, 22:35,
    20:30 , 17: 30, 14:25 , 10:25 ,
    8: 20, 7:20 , 0: 0
}

# File paths
original_location_file = "Location_file.csv"
distance_file = "Distance_Matrix.csv"
travel_time_file = "Time_Matrix.csv"
clustered_location_file = "Clustering_BLG_4.csv"
final_routes_file = "Filtering_BLG_4.csv"
assignment_output_file = "Output_BLG_4.csv"
expanded_schedule_file = "Expanded_Schedule_BLG_4.csv"

# =============================================================================
# II. Helper Functions
# =============================================================================

def calculate_bearing(lat1, lon1, lat2, lon2):
    delta_lon = np.radians(lon2 - lon1)
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    x = np.sin(delta_lon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(delta_lon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

def assign_vehicle_length(total_demand):
    if total_demand > 2200: return 40
    elif total_demand > 1550: return 32
    elif total_demand > 1325: return 24
    elif total_demand > 1255: return 22
    elif total_demand > 893: return 20
    elif total_demand > 686: return 17
    elif total_demand > 400: return 14
    elif total_demand > 250: return 10
    elif total_demand > 180: return 8
    elif total_demand > 0: return 7
    return 0

def find_best_combination(remaining_hubs, available_routes, memo):
    if not remaining_hubs:
        return 0, []
    state = tuple(sorted(list(remaining_hubs)))
    if state in memo: return memo[state]

    best_cost = float('inf')
    best_set = []
    target_hub = next(iter(remaining_hubs))

    # Pruning: Only check routes that contain the current target hub
    potential_routes = [r for r in available_routes if target_hub in r['hubs_set']]
    # Heuristic: Sort by cost to find cheaper branches first
    potential_routes.sort(key=lambda x: x['monthly_cost'])

    for route in potential_routes:
        if route['hubs_set'].issubset(remaining_hubs):
            new_hubs = remaining_hubs - route['hubs_set']
            cost_of_rest, set_of_rest = find_best_combination(new_hubs, available_routes, memo)
            if cost_of_rest != float('inf'):
                total_cost = route['monthly_cost'] + cost_of_rest
                if total_cost < best_cost:
                    best_cost = total_cost
                    best_set = [route] + set_of_rest
    memo[state] = (best_cost, best_set)
    return best_cost, best_set

# =============================================================================
# III. Steps 1 to 9
# =============================================================================

def run_routing_pipeline():
    # --- STEP 1: Dynamic Clustering ---
    print("Step 1: Dynamic Clustering...")
    locs = pd.read_csv(original_location_file)
    # Convert relevant columns to numeric, coercing errors to NaN
    numeric_cols = ['demand', 'ML', 'Freq_Allowed', 'depot_departure (minutes)', 'time_window_end (minutes)', 'latitude', 'longitude']
    for col in numeric_cols:
        if col in locs.columns:
            locs[col] = pd.to_numeric(locs[col], errors='coerce')
            # Fill NaN values if appropriate, e.g., with 0 or a sensible default.
            # For 'demand', 'ML', 'Freq_Allowed', 0 is a reasonable default if missing or invalid.
            if col in ['demand', 'ML', 'Freq_Allowed']:
                locs[col] = locs[col].fillna(0)
            # For latitude and longitude, fill with 0 or drop rows with NaN if appropriate.
            elif col in ['latitude', 'longitude']:
                locs[col] = locs[col].fillna(0) # Or choose a more appropriate strategy

    depot = locs.iloc[0]
    destinations = locs.iloc[1:].copy()
    destinations['bearing'] = destinations.apply(lambda r: calculate_bearing(depot['latitude'], depot['longitude'], r['latitude'], r['longitude']), axis=1)
    destinations = destinations.sort_values('bearing').reset_index(drop=True)
    n = len(destinations)
    for k in range(1, n + 1):
        group_ids = np.array_split(np.arange(n), k)
        labels = np.zeros(n, dtype=int)
        for gid, idxs in enumerate(group_ids): labels[idxs] = gid
        total_comb = sum([math.perm(len(idxs), min(len(idxs), max_hops)) for idxs in group_ids])
        if total_comb <= max_allowed_combinations:
            destinations['bearing_group'] = labels
            break
    destinations['final_group'] = destinations['depot_departure (minutes)'].astype(str) + "-" + destinations['bearing_group'].astype(str)
    clustered_df = pd.concat([pd.DataFrame([depot]), destinations], ignore_index=True)
    clustered_df.to_csv(clustered_location_file, index=False)

    # --- STEP 2: Generate All Route Permutations ---
    print("Step 2: Generating Routes & Pruning Identical Sets...")
    dist_df = pd.read_csv(distance_file)
    dist_dict = {(str(r['location 1']), str(r['location 2'])): r['distance (km)'] for _, r in dist_df.iterrows()}
    attr = clustered_df.set_index('location_name').to_dict('index')
    depot_name = str(clustered_df.iloc[0]['location_name'])

    raw_routes = []
    groups = destinations['final_group'].unique()
    for gid in groups:
        grp_hubs = destinations[destinations['final_group'] == gid]['location_name'].astype(str).tolist()
        for h in range(1, max_hops + 1):
            for perm in itertools.permutations(grp_hubs, h):
                route_seq = [depot_name] + list(perm) + [depot_name]
                d_total = 0
                path_ok = True
                for i in range(len(route_seq)-1):
                    d = dist_dict.get((route_seq[i], route_seq[i+1]))
                    if d is None: path_ok = False; break
                    d_total += d
                if path_ok:
                    raw_routes.append({'route_sequence': " -> ".join(route_seq), 'hubs': list(perm), 'hubs_set': set(perm), 'dist': d_total, 'group': gid})

    # --- STEP 3: Absolute Costing ---
    print("Step 3: Calculating Absolute Costs & Domination Pruning...")
    costed_map = {} # Keyed by hub-set to keep ONLY the cheapest version of a set
    for r in raw_routes:
        base_demand = sum(attr[h]['demand'] for h in r['hubs'])
        max_ml = min(attr[h]['ML'] for h in r['hubs'])
        freq_ok = all(attr[h].get('Freq_Allowed', 0) == 1 for h in r['hubs'])

        c1 = (r['dist'] * VEHICLE_COST_PER_KM.get(assign_vehicle_length(base_demand), 999)) * 30 if assign_vehicle_length(base_demand) <= max_ml else float('inf')
        c2 = (r['dist'] * VEHICLE_COST_PER_KM.get(assign_vehicle_length(base_demand*2), 999)) * 15 if freq_ok and assign_vehicle_length(base_demand*2) <= max_ml else float('inf')

        if c1 == float('inf') and c2 == float('inf'): continue

        m_cost = min(c1, (1.1*c2))
        f_val = 2 if (1.1*c2) < c1 else 1
        t_dem = base_demand * f_val
        v_len = assign_vehicle_length(t_dem)

        h_key = tuple(sorted(list(r['hubs_set'])))
        if h_key not in costed_map or m_cost < costed_map[h_key]['monthly_cost']:
            r.update({'monthly_cost': m_cost, 'Freq': f_val, 'total_demand': t_dem, 'assigned_vehicle_length': v_len})
            costed_map[h_key] = r

    costed_routes = list(costed_map.values())
    pd.DataFrame(costed_routes).to_csv(final_routes_file, index=False)

    # --- STEP 4: Global Selection ---
    print("Step 4: Global Minimum Cost Selection (Set Cover)...")
    final_assigned = []
    for gid in groups:
        grp_routes = [r for r in costed_routes if r['group'] == gid]
        grp_hubs = set(destinations[destinations['final_group'] == gid]['location_name'].astype(str).tolist())

        print(f"Processing Group {gid}: {len(grp_hubs)} hubs, {len(grp_routes)} potential routes...")

        cost, best_set = find_best_combination(grp_hubs, grp_routes, {})

        if not best_set and grp_hubs:
            print(f"CRITICAL ERROR: No valid combination found for Group {gid}!")
            # Check if any hub in this group is missing from ALL grp_routes
            all_covered_hubs = set().union(*(r['hubs_set'] for r in grp_routes))
            missing = grp_hubs - all_covered_hubs
            if missing:
                print(f"Reason: The following hubs have NO valid routes: {missing}")
        else:
            print(f"Success: Group {gid} optimized with {len(best_set)} routes.")
            final_assigned.extend(best_set)

    if not final_assigned:
        print("CRITICAL: final_assigned is completely empty. File will not be written.")
    else:
        assigned_df = pd.DataFrame(final_assigned)
        assigned_df.to_csv(assignment_output_file, index=False)

    # --- STEP 6-9: Timing, Expansion & Missing Columns ---
    print("Step 6: Time Shifting & Metadata Restoration...")
    time_df = pd.read_csv(travel_time_file)
    time_dict = {(str(r['location 1']), str(r['location 2'])): r['travel_time (minutes)'] for _, r in time_df.iterrows()}

    expanded_rows = []
    final_assignment_rows = []

    for idx, row in enumerate(final_assigned):
        stops = [s.strip() for s in row['route_sequence'].split("->")]
        base_dep = attr[stops[1]]['depot_departure (minutes)']

        # Calculate Times
        temp_t = base_dep
        r_times = []
        for i in range(len(stops)-1):
            arr = temp_t + time_dict.get((stops[i], stops[i+1]), 0)
            if stops[i+1] != depot_name:
                service = 120 # Fixed service time
                r_times.append({'DH': stops[i+1], 'arr': arr, 'dep': arr + service})
                temp_t = arr + service
            else: r_times.append({'DH': stops[i+1], 'arr': arr, 'dep': np.nan})

        # Shift to remove idle time
        buffers = [attr[t['DH']]['time_window_end (minutes)'] - t['arr'] for t in r_times if t['DH'] != depot_name]
        shift = max(0, min(buffers)) if buffers else 0

        # Build String Timestamps for Final_Assignment
        arr_str = ":".join([str(round(t['arr'] + shift, 2)) for t in r_times])
        dep_str = ":".join([str(round(t['dep'] + shift, 2)) for t in r_times if not np.isnan(t['dep'])])

        # Append to Final_Assignment (Summary)
        row_summary = row.copy()
        row_summary.update({
            'Route_ID': idx + 1,
            'arrival_times': arr_str,
            'departure_times': dep_str,
            'updated_depot_departure': round(base_dep + shift, 2)
        })
        # Remove sets for CSV cleaniless
        if 'hubs_set' in row_summary: del row_summary['hubs_set']
        final_assignment_rows.append(row_summary)

        # Append to Expanded_Schedule (Detail)
        for i, stop in enumerate(stops):
            expanded_rows.append({
                'Route_ID': idx + 1,
                'Location': stop,
                'Arrival_Time': np.nan if i == 0 else round(r_times[i-1]['arr'] + shift, 2),
                'Departure_Time': round(base_dep + shift, 2) if i == 0 else (round(r_times[i-1]['dep'] + shift, 2) if i < len(stops)-1 else np.nan),
                'Freq': row['Freq'],
                'Vehicle_Length': row['assigned_vehicle_length'],
                'Total_Demand': row['total_demand'],
                'Route_Sequence': row['route_sequence']
            })

    # Save Files
    pd.DataFrame(final_assignment_rows).to_csv(assignment_output_file, index=False)
    pd.DataFrame(expanded_rows).to_csv(expanded_schedule_file, index=False)

    print(f"Pipeline Complete. Summary saved to {assignment_output_file}")
    print(f"Expanded Schedule saved to {expanded_schedule_file}")

if __name__ == "__main__":
    run_routing_pipeline()

Step 1: Dynamic Clustering...
Step 2: Generating Routes & Pruning Identical Sets...
Step 3: Calculating Absolute Costs & Domination Pruning...
Step 4: Global Minimum Cost Selection (Set Cover)...
Processing Group 20.0-0: 2 hubs, 3 potential routes...
Success: Group 20.0-0 optimized with 2 routes.
Processing Group 36.0-0: 1 hubs, 1 potential routes...
Success: Group 36.0-0 optimized with 1 routes.
Processing Group 24.0-0: 2 hubs, 3 potential routes...
Success: Group 24.0-0 optimized with 1 routes.
Processing Group 7.0-0: 2 hubs, 3 potential routes...
Success: Group 7.0-0 optimized with 1 routes.
Processing Group 6.0-0: 1 hubs, 1 potential routes...
Success: Group 6.0-0 optimized with 1 routes.
Processing Group 29.0-0: 2 hubs, 3 potential routes...
Success: Group 29.0-0 optimized with 2 routes.
Processing Group 26.0-0: 3 hubs, 7 potential routes...
Success: Group 26.0-0 optimized with 2 routes.
Processing Group 3.0-0: 3 hubs, 7 potential routes...
Success: Group 3.0-0 optimized with 1 ro

In [ ]:
import os
import pandas as pd # Ensure pandas is imported

clustering_df = pd.read_csv("Clustering_BLG_4.csv")
routes_df = pd.read_csv("Filtering_BLG_4.csv")

# Handle Output_BLG_4.csv
assigned_file_path = "Output_BLG_4.csv"
assigned_df = pd.DataFrame() # Initialize to empty DataFrame
if os.path.exists(assigned_file_path):
    try:
        assigned_df = pd.read_csv(assigned_file_path)
    except pd.errors.EmptyDataError:
        print(f"Warning: {assigned_file_path} is empty or contains no parseable data. Creating an empty DataFrame for 'assigned_df'.")
    except Exception as e:
        print(f"Error reading {assigned_file_path}: {e}. Creating an empty DataFrame for 'assigned_df'.")
else:
    print(f"Warning: {assigned_file_path} does not exist. Creating an empty DataFrame for 'assigned_df'.")

# Handle Expanded_Schedule_BLG_4.csv
expanded_file_path = "Expanded_Schedule_BLG_4.csv"
expanded_schedule_df = pd.DataFrame() # Initialize to empty DataFrame
if os.path.exists(expanded_file_path):
    try:
        expanded_schedule_df = pd.read_csv(expanded_file_path)
    except pd.errors.EmptyDataError:
        print(f"Warning: {expanded_file_path} is empty or contains no parseable data. Creating an empty DataFrame for 'expanded_schedule_df'.")
    except Exception as e:
        print(f"Error reading {expanded_file_path}: {e}. Creating an empty DataFrame for 'expanded_schedule_df'.")
else:
    print(f"Warning: {expanded_file_path} does not exist. Creating an empty DataFrame for 'expanded_schedule_df'.")

# You may need to create these worksheets once
# spreadsheet.add_worksheet(title="Clustering_Output", rows=1000, cols=20)
# spreadsheet.add_worksheet(title="Filtered_Routes", rows=1000, cols=20)
# spreadsheet.add_worksheet(title="Final_Assignment", rows=1000, cols=20)

set_with_dataframe(spreadsheet.worksheet("Clustering_Output"), clustering_df)
set_with_dataframe(spreadsheet.worksheet("Filtered_Routes"), routes_df)
set_with_dataframe(spreadsheet.worksheet("Final_Assignment"), assigned_df)
set_with_dataframe(spreadsheet.worksheet("Expanded_Schedule"), expanded_schedule_df)